In [ ]:
!pip install --upgrade pip setuptools wheel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 44.8 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 80.10.2
    Uninstalling setuptools-80.10.2:
      Successfully uninstalled setuptools-80.10.2
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.


In [ ]:
!pip install transformers datasets seqeval accelerate -q

In [ ]:
import json

with open("ner_colab_export.json") as f:
    data = json.load(f)

label_list = data["label_list"]
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}

print(f"{len(label_list)} labels, {len(data['train'])} train / {len(data['dev'])} dev / {len(data['test'])} test examples")

25 labels, 760 train / 134 dev / 156 test examples


In [ ]:
from datasets import Dataset, DatasetDict

def build(records):
    return Dataset.from_dict({
        "tokens": [r["tokens"] for r in records],
        "ner_tags": [r["ner_tags"] for r in records],
    })

dataset = DatasetDict({
    "train": build(data["train"]),
    "dev": build(data["dev"]),
    "test": build(data["test"]),
})
dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 760
    })
    dev: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 134
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 156
    })
})

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"  # uncased: your text is ALL CAPS,
# so casing carries no signal — uncased is smaller/faster with no real
# information loss here

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align(examples):
    tokenized = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word = None
        aligned = []
        for wid in word_ids:
            if wid is None:
                aligned.append(-100)
            elif wid != prev_word:
                aligned.append(labels[wid])
            else:
                aligned.append(-100)
            prev_word = wid
        all_labels.append(aligned)
    tokenized["labels"] = all_labels
    return tokenized

tokenized_dataset = dataset.map(tokenize_and_align, batched=True)

Map:   0%|          | 0/760 [00:00<?, ? examples/s]

Map:   0%|          | 0/134 [00:00<?, ? examples/s]

Map:   0%|          | 0/156 [00:00<?, ? examples/s]

In [ ]:
import numpy as np
from transformers import (
    AutoModelForTokenClassification, TrainingArguments, Trainer,
    DataCollatorForTokenClassification, EarlyStoppingCallback,
)
from seqeval.metrics import classification_report, f1_score

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=2)

    true_labels, true_preds = [], []
    for pred, label in zip(preds, labels):
        tl, tp = [], []
        for p, l in zip(pred, label):
            if l != -100:
                tl.append(id2label[l])
                tp.append(id2label[p])
        true_labels.append(tl)
        true_preds.append(tp)

    return {"f1": f1_score(true_labels, true_preds)}

args = TrainingArguments(
    output_dir="./ner_checkpoint",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["dev"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1
1,No log,1.319769,0.243415
2,No log,0.897206,0.462011
3,No log,0.751782,0.532649
4,No log,0.702694,0.565251
5,No log,0.678675,0.586411
6,No log,0.678174,0.594946
7,No log,0.672274,0.589520
8,No log,0.676624,0.592226
9,No log,0.681254,0.588402


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=432, training_loss=0.7142543792724609, metrics={'train_runtime': 2950.3717, 'train_samples_per_second': 3.864, 'train_steps_per_second': 0.244, 'total_flos': 104796727910832.0, 'train_loss': 0.7142543792724609, 'epoch': 9.0})

In [ ]:
import pandas as pd

test_commodities = [r["commodity_type"] for r in data["test"]]

preds_output = trainer.predict(tokenized_dataset["test"])
preds = np.argmax(preds_output.predictions, axis=2)
labels = preds_output.label_ids

per_sentence_true, per_sentence_pred = [], []
for pred, label in zip(preds, labels):
    tl, tp = [], []
    for p, l in zip(pred, label):
        if l != -100:
            tl.append(id2label[l])
            tp.append(id2label[p])
    per_sentence_true.append(tl)
    per_sentence_pred.append(tp)

for commodity in sorted(set(test_commodities)):
    idx = [i for i, c in enumerate(test_commodities) if c == commodity]
    c_true = [per_sentence_true[i] for i in idx]
    c_pred = [per_sentence_pred[i] for i in idx]
    print(f"\n{'='*60}\n{commodity.upper()}\n{'='*60}")
    print(classification_report(c_true, c_pred, digits=3))

print(f"\n{'='*60}\nOVERALL\n{'='*60}")
print(classification_report(per_sentence_true, per_sentence_pred, digits=3))

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



GAS_DISTRIBUTION
                    precision    recall  f1-score   support

      ACTION_TAKEN      0.550     0.262     0.355        42
      CAUSE_FACTOR      0.000     0.000     0.000        10
       CONSEQUENCE      0.619     0.667     0.642        39
         DATE_TIME      0.545     0.600     0.571        20
         EQUIPMENT      0.691     0.810     0.746        58
      FAILURE_MODE      0.522     0.444     0.480        27
INSPECTION_FINDING      0.000     0.000     0.000         1
          LOCATION      0.438     0.424     0.431        33
     MATERIAL_SPEC      0.758     0.862     0.806        29
        PARTY_ROLE      0.491     0.634     0.553        41
          QUANTITY      0.870     0.833     0.851        24
    REGULATORY_REF      0.000     0.000     0.000         2

         micro avg      0.609     0.592     0.600       326
         macro avg      0.457     0.461     0.453       326
      weighted avg      0.582     0.592     0.578       326


GAS_TRANSMISSION_G

/usr/local/lib/python3.13/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
trainer.save_model("./phmsa_ner_model")
tokenizer.save_pretrained("./phmsa_ner_model")

!zip -r phmsa_ner_model.zip phmsa_ner_model
from google.colab import files
files.download("phmsa_ner_model.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: phmsa_ner_model/ (stored 0%)
  adding: phmsa_ner_model/tokenizer.json (deflated 71%)
  adding: phmsa_ner_model/model.safetensors (deflated 8%)
  adding: phmsa_ner_model/config.json (deflated 64%)
  adding: phmsa_ner_model/tokenizer_config.json (deflated 43%)
  adding: phmsa_ner_model/training_args.bin (deflated 53%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>